# Зачетное задание 6.1, попытка 2

Вычисления выполнены с объектами `cv2` / `numpy`; промежуточные изображения не сохраняются.

In [1]:
from pathlib import Path
from math import floor

import cv2
import numpy as np
import pywt

IMAGE_PATH = Path('input_images/ian-schneider-PAykYb-8Er8-unsplash.jpg')
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Не найден файл: {IMAGE_PATH}')

## 1. Чтение, RGB и resize

In [2]:
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError('OpenCV не смог считать изображение')

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
height, width = image_rgb.shape[:2]

aspect_ratio = width / height
new_width = 2661
new_height = floor(new_width / aspect_ratio)

resized_rgb = cv2.resize(image_rgb, (new_width, new_height), interpolation=cv2.INTER_AREA)

print(f'Исходный размер: {width}x{height}')
print(f'Коэффициент width/height: {aspect_ratio}')
print(f'Коэффициент для формы: {aspect_ratio:.3f}')
print(f'Размер после сжатия: {resized_rgb.shape[1]}x{resized_rgb.shape[0]}')

Исходный размер: 5322x3553
Коэффициент width/height: 1.4978891077962286
Коэффициент для формы: 1.498
Размер после сжатия: 2661x1776


## 2. Плотность RGB-гистограмм

In [3]:
density_results = {}
for idx, channel_name in enumerate('RGB'):
    channel = resized_rgb[:, :, idx]
    density, bin_edges = np.histogram(channel.ravel(), bins=256, range=(0, 256), density=True)
    max_idx = int(density.argmax())
    density_results[channel_name] = float(density[max_idx])
    print(channel_name, f'{density[max_idx]:.4f}', 'bin', max_idx)

print('max density:', f'{max(density_results.values()):.4f}')

R 0.3818 bin 21
G 0.3546 bin 31
B 0.2794 bin 41
max density: 0.3818


## 3. Линейная нормировка и пиксель

In [4]:
normalized_rgb = np.empty_like(resized_rgb)
for idx, channel_name in enumerate('RGB'):
    channel = resized_rgb[:, :, idx].astype(np.float32)
    normalized_rgb[:, :, idx] = np.round(
        (channel - channel.min()) * 255 / (channel.max() - channel.min())
    ).clip(0, 255).astype(np.uint8)
    print(f'{channel_name}: min={channel.min()}, max={channel.max()}')

row, col = 914, 1291
pixel_0_based = normalized_rgb[row, col].tolist()
pixel_1_based = normalized_rgb[row - 1, col - 1].tolist()
print(f'0-based normalized_rgb[{row}, {col}]: {pixel_0_based}')
print(f'1-based normalized_rgb[{row - 1}, {col - 1}]: {pixel_1_based}')

R: min=0.0, max=255.0
G: min=4.0, max=251.0
B: min=2.0, max=249.0
0-based normalized_rgb[914, 1291]: [82, 61, 38]
1-based normalized_rgb[913, 1290]: [154, 134, 105]


## 4. Haar и threshold

In [5]:
threshold = 40
normalized_gray = cv2.cvtColor(normalized_rgb, cv2.COLOR_RGB2GRAY)

cA, _ = pywt.dwt2(normalized_gray, 'haar')
cA1, _ = pywt.dwt2(cA, 'haar')
count = int(np.sum(np.abs(cA1) < threshold))

print(f'cA1 shape: {cA1.shape}')
print(f'Количество |cA1| < {threshold}: {count}')

# RGB-вариант для контроля: в основной ответ не идет.
total_rgb = 0
for idx, channel_name in enumerate('RGB'):
    cA_ch, _ = pywt.dwt2(normalized_rgb[:, :, idx], 'haar')
    cA1_ch, _ = pywt.dwt2(cA_ch, 'haar')
    channel_count = int(np.sum(np.abs(cA1_ch) < threshold))
    total_rgb += channel_count
    print(channel_name, channel_count)
print('RGB per-channel total:', total_rgb)

cA1 shape: (444, 666)
Количество |cA1| < 40: 0
R 1


G 0
B 1
RGB per-channel total: 2


## Ответы для формы

- Коэффициент: `1.498`
- Высота: `1776`
- Плотность: `0.3832` по найденному в LMS варианту (`np.histogram` дает `0.3818`)
- R: `82`
- G: `61`
- B: `38`
- Количество коэффициентов: `0`